# Notebook 3: Sensitivity Flagging — Method 2 (LLaVA Direct Classification)

This notebook classifies TIFF images for sensitive content using **LLaVA** (Large Language and Vision Assistant) running locally via **Ollama**.

### Pipeline
1. Collect TIFF images from the input folder
2. Convert each TIFF to JPEG (max 4000px), resizing if needed
3. Send each image to LLaVA with a structured moderation prompt
4. Parse the JSON response for: `offensive`, `category`, `rationale`
5. Save results to CSV with intermediate saves every 100 images

### Sensitive Categories Detected
| Category |
|----------|
| Historical racialized performance |
| Human remains |
| Native American imagery |
| Nudity/sexual content |
| Violence/graphic content |
| Hate symbols |
| Medical/health records |
| Student/PII records |
| Other sensitive categories |

### Prerequisites
- [Ollama](https://ollama.com) installed and running
- `llava-llama3` model pulled: `ollama pull llava-llama3`

### Performance Notes
- ~2–2.5 minutes per image (model inference is the bottleneck, not image size)
- ~33–40 hours projected for 1,000 images
- **Finding:** LLaVA flagged all images as non-offensive in testing — may require fine-tuning for historical archival content

## Configuration

In [ ]:
import os
import platform

# ============================================================
# CONFIGURATION — Edit these before running
# ============================================================

# Path to Ollama executable
if platform.system() == "Windows":
    OLLAMA_PATH = r"C:\Users\YOUR_USERNAME\AppData\Local\Programs\Ollama\ollama.exe"
else:
    OLLAMA_PATH = "/usr/local/bin/ollama"  # or wherever ollama is installed on Mac/Linux

# Ollama model to use
MODEL_NAME = "llava-llama3"

# Folder containing TIFF images
IMAGE_FOLDER = r"path/to/your/tiff/images"  # e.g. r"C:\Downloads\extracted_scans\Numbered Scans_1"

# Output CSV path
OUTPUT_CSV = r"path/to/output/llava_classification_results.csv"

# JPEG conversion settings
MAX_JPEG_DIM = 4000  # Max pixel dimension for JPEG conversion
JPEG_QUALITY = 85

# Range of files to process (adjust to resume or process a subset)
START_INDEX = 0      # 0 = start from the beginning
END_INDEX = 1000     # Set to None to process all files

# Save intermediate results every N images (guards against crashes)
INTERMEDIATE_SAVE = 100

print(f"Ollama path: {OLLAMA_PATH}")
print(f"Model: {MODEL_NAME}")
print(f"Image folder: {IMAGE_FOLDER}")
print(f"Output CSV: {OUTPUT_CSV}")

## Imports & Category Definitions

In [ ]:
import json
import csv
import time
import subprocess
from PIL import Image, ImageFile, UnidentifiedImageError

# Allow very large images and truncated files
Image.MAX_IMAGE_PIXELS = None
ImageFile.LOAD_TRUNCATED_IMAGES = True

CATEGORIES = [
    "Historical racialized performance",
    "Human remains",
    "Native American imagery",
    "Nudity/sexual content",
    "Violence/graphic content",
    "Hate symbols",
    "Medical/health records",
    "Student/PII records",
    "Other sensitive categories",
]

print(f"Monitoring {len(CATEGORIES)} sensitive content categories.")

## Helper Functions

In [ ]:
def tiff_to_jpg(tif_path: str, jpg_path: str, max_dim: int = 4000, quality: int = 85):
    """Convert a TIFF to JPEG, resizing if the longest dimension exceeds max_dim."""
    with Image.open(tif_path) as img:
        if getattr(img, "n_frames", 1) > 1:
            img.seek(0)  # Use first frame for multi-frame TIFFs
        img = img.convert("RGB")
        w, h = img.size
        scale = max(w / max_dim, h / max_dim, 1.0)
        if scale > 1.0:
            new_size = (max(1, int(round(w / scale))), max(1, int(round(h / scale))))
            img = img.resize(new_size, Image.LANCZOS)
        os.makedirs(os.path.dirname(jpg_path), exist_ok=True)
        img.save(jpg_path, "JPEG", quality=quality, optimize=True)


def run_ollama(prompt: str) -> str:
    """Send a prompt to Ollama and return the response text."""
    result = subprocess.run(
        [OLLAMA_PATH, "run", MODEL_NAME],
        input=prompt.encode("utf-8"),
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        creationflags=getattr(subprocess, "CREATE_NO_WINDOW", 0),
    )
    stdout = result.stdout.decode("utf-8", errors="replace").strip()
    stderr = result.stderr.decode("utf-8", errors="replace").strip()
    return stdout or stderr


def build_classification_prompt(jpg_path: str) -> str:
    """Build the moderation prompt for a given image path."""
    categories_list = ", ".join(CATEGORIES)
    return (
        f"<image>{jpg_path}</image>\n\n"
        "You are an image moderation classifier.\n"
        "Determine if the image contains any hate symbols or offensive/insensitive content.\n"
        f"If yes, assign it to ONE of these categories:\n{categories_list}\n\n"
        "If not offensive, set category and rationale to null.\n\n"
        "Output format (strict JSON only):\n"
        '{"offensive": true/false, "category": "category_name_or_null", "rationale": "brief_reason_or_null"}'
    )

## Run Classification

In [ ]:
# Collect and sort TIFF files
tif_files = sorted(
    f for f in os.listdir(IMAGE_FOLDER)
    if f.lower().endswith(('.tif', '.tiff'))
)

# Apply start/end slice
tif_files = tif_files[START_INDEX:END_INDEX]

if not tif_files:
    print("No TIFF files found. Check IMAGE_FOLDER path.")
    raise SystemExit(0)

print(f"Processing {len(tif_files)} TIFF files...")

# Create folder for JPEG conversions
jpg_folder = os.path.join(IMAGE_FOLDER, "jpg_converted")
os.makedirs(jpg_folder, exist_ok=True)

results = []
batch_start = time.time()

for idx, tif_file in enumerate(tif_files, start=1):
    start_time = time.time()

    tif_path = os.path.join(IMAGE_FOLDER, tif_file)
    jpg_path = os.path.join(jpg_folder, os.path.splitext(tif_file)[0] + '.jpg')

    # Convert TIFF to JPEG if not already done
    if not os.path.exists(jpg_path):
        try:
            tiff_to_jpg(tif_path, jpg_path, max_dim=MAX_JPEG_DIM, quality=JPEG_QUALITY)
        except (UnidentifiedImageError, OSError, RuntimeError) as e:
            print(f"[{idx}/{len(tif_files)}] Conversion failed for {tif_file}: {e}")
            continue

    # Build and send prompt
    prompt = build_classification_prompt(jpg_path)
    response = run_ollama(prompt)

    # Parse JSON response
    try:
        parsed = json.loads(response)
        offensive = parsed.get("offensive", False)
        category = parsed.get("category") if offensive else None
        rationale = parsed.get("rationale", parsed.get("ratiole"))  # handle common typo in model output
        results.append({
            "filename": tif_file,
            "offensive": offensive,
            "category": category,
            "rationale": rationale,
        })
    except json.JSONDecodeError:
        # Model returned non-JSON — store raw response in rationale
        results.append({
            "filename": tif_file,
            "offensive": None,
            "category": None,
            "rationale": response.strip(),
        })

    elapsed = time.time() - start_time
    print(f"[{idx}/{len(tif_files)}] '{tif_file}' — {elapsed:.2f}s")

    # Intermediate save
    if idx % INTERMEDIATE_SAVE == 0:
        with open(OUTPUT_CSV, "w", newline='', encoding="utf-8") as f:
            writer = csv.DictWriter(f, fieldnames=["filename", "offensive", "category", "rationale"])
            writer.writeheader()
            writer.writerows(results)
        print(f"  → Intermediate save at image {idx}")

# Final save
with open(OUTPUT_CSV, "w", newline='', encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["filename", "offensive", "category", "rationale"])
    writer.writeheader()
    writer.writerows(results)

total_elapsed = time.time() - batch_start
flagged = sum(1 for r in results if r["offensive"])
print(f"\nCompleted {len(tif_files)} images in {total_elapsed/60:.2f} minutes")
print(f"Flagged as offensive: {flagged}")
print(f"Results saved to: {OUTPUT_CSV}")